# PyTorch Data Splitting Examples

This notebook demonstrates various methods for splitting and sampling data in PyTorch.


## Resources


**Dataset**

- https://www.youtube.com/watch?v=Sj-gIb0QiRM
- https://www.youtube.com/watch?v=PXOzkkB5eH0&list=PLqnslRFeH2UrcDBWF5mfPGpqQDSta6VK4&index=9

**Splitting**

- https://docs.pytorch.org/docs/stable/data.html
- https://discuss.pytorch.org/t/how-to-split-dataset-into-test-and-validation-sets/33987/2
- https://stackoverflow.com/questions/50544730/how-do-i-split-a-custom-dataset-into-training-and-test-datasets


In [8]:
import numpy as np
import torch
from torch.utils.data import (
    ConcatDataset,
    DataLoader,
    Dataset,
    RandomSampler,
    SequentialSampler,
    Subset,
    SubsetRandomSampler,
    random_split,
)

In [9]:
# Create a simple dataset for demonstration
class SimpleDataset(Dataset):
    def __init__(self, size=100):
        self.data = torch.randn(size, 3)  # Random data with 3 features
        self.labels = torch.randint(0, 2, (size,))  # Binary labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# Create dataset instance
dataset = SimpleDataset(100)
print(f"Dataset size: {len(dataset)}")

Dataset size: 100


## 1. random_split

Randomly splits a dataset into non-overlapping new datasets. Commonly used for train/validation/test splits.


In [10]:
# Split dataset into train (70%), validation (15%), test (15%)
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

Train size: 70
Validation size: 15
Test size: 15


## 2. Subset

Creates a subset of a dataset using specific indices. Useful when you need precise control over which samples to include.


In [11]:
# Create a subset using specific indices
indices = [0, 5, 10, 15, 20, 25]  # Select specific samples
subset_dataset = Subset(dataset, indices)

print(f"Original dataset size: {len(dataset)}")
print(f"Subset size: {len(subset_dataset)}")
print(f"Subset indices: {indices}")

# Create data loader for subset
subset_loader = DataLoader(subset_dataset, batch_size=4, shuffle=True)

# Show first batch
for batch_idx, (data, labels) in enumerate(subset_loader):
    print(f"Batch {batch_idx}: data shape = {data.shape}, labels = {labels}")
    break

Original dataset size: 100
Subset size: 6
Subset indices: [0, 5, 10, 15, 20, 25]
Batch 0: data shape = torch.Size([4, 3]), labels = tensor([0, 1, 0, 0])


## 3. ConcatDataset

Concatenates multiple datasets into a single dataset. Useful for combining datasets from different sources.


In [12]:
# Create two separate datasets
dataset1 = SimpleDataset(50)
dataset2 = SimpleDataset(30)

# Concatenate them
combined_dataset = ConcatDataset([dataset1, dataset2])

print(f"Dataset 1 size: {len(dataset1)}")
print(f"Dataset 2 size: {len(dataset2)}")
print(f"Combined dataset size: {len(combined_dataset)}")

# Create data loader for combined dataset
combined_loader = DataLoader(combined_dataset, batch_size=8, shuffle=True)

Dataset 1 size: 50
Dataset 2 size: 30
Combined dataset size: 80


## 4. RandomSampler

Samples elements randomly without replacement. Used for random sampling when you want to control the sampling process explicitly.


In [13]:
# Create a RandomSampler
random_sampler = RandomSampler(dataset)

# Create data loader with random sampler
random_loader = DataLoader(dataset, batch_size=8, sampler=random_sampler)

print("Random sampling order (first 20 indices):")
sampler_iter = iter(random_sampler)
for i in range(20):
    print(next(sampler_iter), end=" ")
print("\n")

Random sampling order (first 20 indices):
11 33 96 72 75 90 93 24 19 39 86 31 17 74 70 30 18 98 91 29 



## 5. SequentialSampler

Samples elements sequentially in order. Used when you need deterministic, ordered sampling - particularly useful for time series data where temporal order must be preserved.


In [14]:
# Time series split - preserving temporal order
dataset_size = len(dataset)
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# Calculate split points for sequential splitting
train_end = int(train_ratio * dataset_size)
val_end = int((train_ratio + val_ratio) * dataset_size)

# Create indices for each split (maintaining order)
train_indices = list(range(0, train_end))
val_indices = list(range(train_end, val_end))
test_indices = list(range(val_end, dataset_size))

print(f"Time series sequential split:")
print(
    f"Train: indices {train_indices[0]}-{train_indices[-1]} ({len(train_indices)} samples)"
)
print(f"Val: indices {val_indices[0]}-{val_indices[-1]} ({len(val_indices)} samples)")
print(
    f"Test: indices {test_indices[0]}-{test_indices[-1]} ({len(test_indices)} samples)"
)

# Create sequential samplers for each split
train_seq_sampler = SequentialSampler(Subset(dataset, train_indices))
val_seq_sampler = SequentialSampler(Subset(dataset, val_indices))
test_seq_sampler = SequentialSampler(Subset(dataset, test_indices))

# Create data loaders with sequential samplers
train_seq_loader = DataLoader(
    Subset(dataset, train_indices), batch_size=8, sampler=train_seq_sampler
)
val_seq_loader = DataLoader(
    Subset(dataset, val_indices), batch_size=8, sampler=val_seq_sampler
)
test_seq_loader = DataLoader(
    Subset(dataset, test_indices), batch_size=8, sampler=test_seq_sampler
)

# Demonstrate that order is preserved
print("\nFirst 10 train indices (sequential):")
train_iter = iter(train_seq_sampler)
for i in range(min(10, len(train_indices))):
    print(next(train_iter), end=" ")
print()

print("First 10 validation indices (sequential):")
val_iter = iter(val_seq_sampler)
for i in range(min(10, len(val_indices))):
    print(next(val_iter), end=" ")
print()

# Verify temporal order is maintained
print(f"\nTemporal order preserved: Train < Val < Test")
print(f"Train max index: {max(train_indices)} < Val min index: {min(val_indices)}")
print(f"Val max index: {max(val_indices)} < Test min index: {min(test_indices)}")

Time series sequential split:
Train: indices 0-69 (70 samples)
Val: indices 70-88 (19 samples)
Test: indices 89-99 (11 samples)

First 10 train indices (sequential):
0 1 2 3 4 5 6 7 8 9 
First 10 validation indices (sequential):
0 1 2 3 4 5 6 7 8 9 

Temporal order preserved: Train < Val < Test
Train max index: 69 < Val min index: 70
Val max index: 88 < Test min index: 89


## 6. SubsetRandomSampler

Randomly samples from a subset of indices without replacement. Useful for creating random samples from specific subsets of your data.


In [15]:
# Define subset indices for train/validation split
dataset_size = len(dataset)
indices = list(range(dataset_size))
validation_split = 0.2  # 20% for validation
random_seed = 42
shuffle_dataset = True

split = int(np.floor(validation_split * dataset_size))
if shuffle_dataset:
    np.random.seed(random_seed)
    np.random.shuffle(indices)

train_indices, val_indices = indices[split:], indices[:split]

print(f"Dataset size: {dataset_size}")
print(f"Train indices: {len(train_indices)} samples")
print(f"Validation indices: {len(val_indices)} samples")

# Create SubsetRandomSamplers for train and validation
train_sampler = SubsetRandomSampler(train_indices)
valid_sampler = SubsetRandomSampler(val_indices)

# Create data loaders with subset random samplers
train_loader = DataLoader(dataset, batch_size=8, sampler=train_sampler)
valid_loader = DataLoader(dataset, batch_size=8, sampler=valid_sampler)

print("First 10 train indices:")
train_iter = iter(train_sampler)
for i in range(10):
    print(next(train_iter), end=" ")
print("\n")

print("First 10 validation indices:")
valid_iter = iter(valid_sampler)
for i in range(10):
    print(next(valid_iter), end=" ")
print()

# Verify no overlap between train and validation sets
print(
    f"No overlap between train/val: {len(set(train_indices) & set(val_indices)) == 0}"
)

Dataset size: 100
Train indices: 80 samples
Validation indices: 20 samples
First 10 train indices:
46 1 64 2 92 91 16 60 32 68 

First 10 validation indices:
30 70 76 80 73 77 53 10 0 18 
No overlap between train/val: True


## Summary

- **random_split**: Splits dataset into non-overlapping subsets (train/val/test)
- **Subset**: Creates subset using specific indices for precise control
- **ConcatDataset**: Combines multiple datasets into one
- **RandomSampler**: Random sampling without replacement
- **SequentialSampler**: Sequential, ordered sampling
- **SubsetRandomSampler**: Random sampling from a specific subset of indices
